## Version 7.0
- Entity to entity comparison

In [1]:
import faiss
import numpy as np
from collections import defaultdict
from sentence_transformers import SentenceTransformer, util
from transformers import pipeline
from fuzzywuzzy import fuzz

from scipy.sparse import lil_matrix
from sklearn.feature_extraction.text import TfidfTransformer
from sklearn.preprocessing import normalize

/Users/user/miniconda3/envs/graphmaker/lib/python3.11/site-packages/fuzzywuzzy/fuzz.py:11: UserWarning: Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning
  warnings.warn('Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning')


In [2]:
class HybridEntityResolver:

    def __init__(self,
                 embedding_model="all-mpnet-base-v2",
                 sim_threshold=0.65):

        # -----------------------
        # Models
        # -----------------------
        self.model = SentenceTransformer(embedding_model)
        self.ner_pipeline = pipeline(
            "ner",
            model="dslim/bert-base-NER",
            aggregation_strategy="simple"
        )

        # -----------------------
        # Entity Storage
        # -----------------------
        self.entity_store = {}
        self.entity_id_counter = 1

        # -----------------------
        # FAISS
        # -----------------------
        self.dimension = 768
        self.faiss_index = faiss.IndexFlatIP(self.dimension)
        self.int2id = {}

        # -----------------------
        # Graph
        # -----------------------
        self.accepted_triples = []
        self.existing_edge_keys = set()

        # -----------------------
        # Relational
        # -----------------------
        self.rel_signatures = None
        self.entity2idx_rel = None
        self.pred2idx = None

        # -----------------------
        self.sim_threshold = sim_threshold

    # ============================================================
    # -------------------- Utility Methods -----------------------
    # ============================================================

    def _normalize(self, v):
        return v / np.linalg.norm(v)

    def _string_sim(self, a, b):
        return fuzz.token_sort_ratio(a, b) / 100

    def _predict_type(self, entity_text, predicate=None):
        text = f"{entity_text} {predicate}" if predicate else entity_text

        try:
            entities = self.ner_pipeline(text)
        except Exception:
            return "UNKNOWN"

        for ent in entities:
            if ent["word"].lower() in entity_text.lower():
                label = ent["entity_group"]
                return {
                    "PER": "PERSON",
                    "ORG": "ORGANIZATION",
                    "LOC": "LOCATION",
                    "MISC": "MISC"
                }.get(label, "UNKNOWN")

        return "UNKNOWN"

    def _types_compatible(self, t1, t2):
        if t1 == "UNKNOWN" or t2 == "UNKNOWN":
            return True
        return t1 == t2

    # ============================================================
    # -------------------- FAISS Blocking ------------------------
    # ============================================================

    def _add_to_faiss(self, entity_id, embedding):
        idx = self.faiss_index.ntotal
        self.faiss_index.add(embedding)
        self.int2id[idx] = entity_id

    def _semantic_blocking(self, entity_text, top_k=10, sim_threshold=0.7):

        if self.faiss_index.ntotal == 0:
            return []

        emb = self.model.encode(entity_text)
        emb = np.asarray(emb, dtype="float32").reshape(1, -1)
        faiss.normalize_L2(emb)

        scores, indices = self.faiss_index.search(emb, top_k)

        candidates = []
        for score, idx in zip(scores[0], indices[0]):
            if idx == -1:
                continue
            if score >= sim_threshold:
                candidates.append(
                    self.int2id[idx]
                )
        return candidates

    # ============================================================
    # -------------------- Relational Model ----------------------
    # ============================================================

    def _rebuild_relational_model(self):

        if len(self.accepted_triples) < 5:
            return

        predicates = set(p for _, p, _ in self.accepted_triples)
        self.pred2idx = {p: i for i, p in enumerate(sorted(predicates))}

        entities = set()
        for s, _, o in self.accepted_triples:
            entities.add(s)
            entities.add(o)

        entity_list = sorted(entities)
        self.entity2idx_rel = {e: i for i, e in enumerate(entity_list)}

        m = len(self.pred2idx)
        n = len(entity_list)

        signatures = lil_matrix((n, 2*m), dtype=np.float32)

        for s, p, o in self.accepted_triples:
            p_idx = self.pred2idx[p]
            signatures[self.entity2idx_rel[s], p_idx] += 1
            signatures[self.entity2idx_rel[o], p_idx + m] += 1

        transformer = TfidfTransformer(norm=None)
        signatures = transformer.fit_transform(signatures)
        self.rel_signatures = normalize(signatures, norm='l2')

    def _relational_similarity(self, id1, id2):

        if self.rel_signatures is None:
            return 0.0

        if id1 not in self.entity2idx_rel or id2 not in self.entity2idx_rel:
            return 0.0

        idx1 = self.entity2idx_rel[id1]
        idx2 = self.entity2idx_rel[id2]

        return self.rel_signatures[idx1].dot(
            self.rel_signatures[idx2].T
        )[0, 0]

    # ============================================================
    # -------------------- Composite Similarity ------------------
    # ============================================================

    def _composite_similarity(self, entity_text, existing_id, predicate):

        # existing = self.entity_store[existing_id]

        # s_sim = self._string_sim(entity_text, existing["name"])

        # emb_query = self.model.encode(entity_text)
        # emb_existing = existing["embedding"]
        # e_sim = util.cos_sim(
        #     emb_query, emb_existing
        # ).item()

        # new_type = self._predict_type(entity_text, predicate)
        # p_sim = 1.0 if self._types_compatible(
        #     new_type, existing["type"]
        # ) else 0.0

        # r_sim = self._relational_similarity(
        #     existing_id, existing_id
        # )

        # return (
        #     0.3 * s_sim +
        #     0.35 * e_sim +
        #     0.1 * p_sim +
        #     0.25 * r_sim
        # )

        existing = self.entity_store[existing_id]

        # 1. String similarity
        s_sim = self._string_sim(entity_text, existing["name"])

        # 2. Contextual embedding similarity
        context_query = f"{entity_text} in relation {predicate}"
        emb_query = self.model.encode(context_query)
        emb_existing = existing["embedding"]

        e_sim = util.cos_sim(emb_query, emb_existing).item()

        # 3. Type similarity
        new_type = self._predict_type(entity_text, predicate)
        p_sim = 1.0 if self._types_compatible(new_type, existing["type"]) else 0.0

        # 4. Relational similarity (only if available)
        r_sim = 0.0
        if self.rel_signatures is not None and existing_id in self.entity2idx_rel:
            r_sim = self._relational_similarity(existing_id, existing_id)

        return (
        0.30 * s_sim +
        0.35 * e_sim +
        0.10 * p_sim +
        0.25 * r_sim
        )

    # ============================================================
    # -------------------- Entity Resolution ---------------------
    # ============================================================

    def resolve_entity(self, entity_text, predicate):

        candidates = self._semantic_blocking(entity_text)

        best_id = None
        best_score = 0

        for candidate_id in candidates:

            score = self._composite_similarity(
                entity_text,
                candidate_id,
                predicate
            )

            if score > best_score:
                best_score = score
                best_id = candidate_id

        if best_id and best_score >= self.sim_threshold:
            return best_id, "MERGE"

        return self._insert_entity(entity_text, predicate)

    def _insert_entity(self, entity_text, predicate):

        new_id = f"E{self.entity_id_counter}"
        self.entity_id_counter += 1

        embedding = self.model.encode(entity_text)
        embedding = np.asarray(embedding, dtype="float32").reshape(1, -1)
        faiss.normalize_L2(embedding)

        self.entity_store[new_id] = {
            "name": entity_text,
            "type": self._predict_type(entity_text, predicate),
            "embedding": embedding,
            "neighbors": set()
        }

        self._add_to_faiss(new_id, embedding)

        return new_id, "INSERT"

    # ============================================================
    # -------------------- Triple Processing ---------------------
    # ============================================================

    def process_triple(self, subject, predicate, obj):

        subj_id, _ = self.resolve_entity(subject, predicate)
        obj_id, _ = self.resolve_entity(obj, predicate)

        edge_key = (subj_id, predicate, obj_id)

        if edge_key in self.existing_edge_keys:
            return

        self.existing_edge_keys.add(edge_key)

        self.entity_store[subj_id]["neighbors"].add(
            f"{predicate}:{obj_id}"
        )
        self.entity_store[obj_id]["neighbors"].add(
            f"{predicate}:{subj_id}"
        )

        self.accepted_triples.append(
            (subj_id, predicate, obj_id)
        )

        if len(self.accepted_triples) % 10 == 0:
            self._rebuild_relational_model()

        
    def get_text_to_entity_mapping(self):
        mapping = {}
        for eid, data in self.entity_store.items():
            mapping[data["name"]] = eid
        return mapping

In [3]:
from knowledge_graph_maker import Edge, Node

list_of_edges = [

    # =========================
    # 1. Homonym & Lexical Collision
    # =========================

    Edge(
        node_1=Node(
            label="Organization",
            name="Apple",
            properties={"entity_type": "technology_company"}
        ),
        node_2=Node(
            label="Product",
            name="iPhone 15",
            properties={}
        ),
        relationship="released",
        metadata={
            "summary": "Apple released the iPhone 15.",
            "subject_gold": "APPLE_TECH",
            "object_gold": "IPHONE_15",
            "generated_at": "2025-10-30 17:34:07.282620"
        },
        order=0
    ),

    Edge(
        node_1=Node(
            label="Food",
            name="Apple",
            properties={"entity_type": "fruit"}
        ),
        node_2=Node(
            label="Time",
            name="September",
            properties={}
        ),
        relationship="harvested in",
        metadata={
            "summary": "Apple is harvested in September.",
            "subject_gold": "APPLE_FRUIT",
            "object_gold": "MONTH_SEPTEMBER",
            "generated_at": "2025-10-30 17:34:07.282620"
        },
        order=1
    ),

    Edge(
        node_1=Node(
            label="Food",
            name="Apple",
            properties={"entity_type": "fruit"}
        ),
        node_2=Node(
            label="FoodCategory",
            name="Fruit",
            properties={}
        ),
        relationship="is a type of",
        metadata={
            "summary": "Apple is a type of fruit.",
            "subject_gold": "APPLE_FRUIT",
            "object_gold": "FRUIT",
            "generated_at": "2025-10-30 17:34:07.282620"
        },
        order=2
    ),

    Edge(
        node_1=Node(
            label="Organization",
            name="Apple Inc.",
            properties={"entity_type": "technology_company"}
        ),
        node_2=Node(
            label="Product",
            name="MacBook",
            properties={}
        ),
        relationship="sells",
        metadata={
            "summary": "Apple Inc. sells the MacBook.",
            "subject_gold": "APPLE_TECH",
            "object_gold": "MACBOOK",
            "generated_at": "2025-10-30 17:34:07.282620"
        },
        order=3
    )
]

toy_data = []
unique_entities = set()
for edge in list_of_edges:
    unique_entities.add(edge.node_1.name)
    unique_entities.add(edge.node_2.name)

    toy_data.append({
        "subject": edge.node_1.name,
        "subject_gold": edge.metadata.get("subject_gold"),
        "predicate": edge.relationship,
        "object": edge.node_2.name,     
        "object_gold": edge.metadata.get("object_gold"),
        # "context": edge.metadata.get("summary"),
    })

In [4]:
resolver = HybridEntityResolver()

for triple in toy_data:
    resolver.process_triple(
        triple["subject"],
        triple["predicate"],
        triple["object"]
    )

Some weights of the model checkpoint at dslim/bert-base-NER were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use mps:0


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

...

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

In [5]:
from collections import defaultdict
from itertools import combinations

# ----------------------------
# Helper: Build entity → gold clusters mapping
# ----------------------------

def build_gold_entity_map(gold_clusters):
    """
    Maps entity_text -> set of gold clusters it belongs to
    """
    gold_entity_map = defaultdict(set)
    for gold_cluster, entities in gold_clusters.items():
        for e in entities:
            gold_entity_map[e].add(gold_cluster)
    return gold_entity_map

# ----------------------------
# Helper: Generate all pairwise links for evaluation
# ----------------------------
def pairwise_links(items):
    """
    Returns all pairwise combinations (unordered) of items.
    Singleton clusters are treated as a self-link so they contribute to metrics.
    """
    if len(items) == 0:
        return set()
    if len(items) == 1:
        return {frozenset(items)}  # singleton contributes one link
    return set(frozenset([a, b]) for a, b in combinations(items, 2))

# ----------------------------
# Evaluation function
# ----------------------------
def evaluate_cluster_alignment(gold_clusters, predicted_clusters):
    gold_entity_map = build_gold_entity_map(gold_clusters)

    results = []

    for pred_cluster_id, pred_entities in predicted_clusters.items():
        matched_gold_clusters = defaultdict(set)

        for entity in pred_entities:
            golds = gold_entity_map.get(entity, set())
            if not golds:
                matched_gold_clusters["UNSEEN"].add(entity)
            else:
                for g in golds:
                    matched_gold_clusters[g].add(entity)

        results.append({
            "predicted_cluster": pred_cluster_id,
            "predicted_entities": pred_entities,
            "matched_gold_clusters": dict(matched_gold_clusters),
        })

    for r in results:
        print("=" * 50)
        print(f"Predicted Cluster: {r['predicted_cluster']}")
        print(f"Entities: {r['predicted_entities']}")

        golds = r["matched_gold_clusters"]

        if len(golds) == 1:
            gold_cluster = next(iter(golds))
            print(f"Pure match with gold cluster: {gold_cluster}")
        elif len(golds) > 1:
            print("Mixed cluster! Matches multiple gold clusters:")
            for g, ents in golds.items():
                print(f"  - {g}: {ents}")
        else:
            print("No gold cluster matched")
   
def evaluate_er_pipeline(resolved_entities, toy_dataset):
    """
    Evaluate entity resolution performance.
    
    resolved_entities: dict mapping entity_text -> resolved_entity_id
    toy_dataset: list of dicts with subject_gold / object_gold
    """

    # -------------------------
    # Build gold clusters (entity-level)
    # -------------------------
    gold_clusters = defaultdict(set)
    for triple in toy_dataset:
        gold_clusters[triple["subject_gold"]].add(triple["subject"])
        gold_clusters[triple["object_gold"]].add(triple["object"])

    print("Gold Clusters:")
    for cluster_id, entities in gold_clusters.items():
        print(f"{cluster_id}: {entities}")

    # -------------------------
    # Build predicted clusters
    # -------------------------
    pred_clusters = defaultdict(set)

    for triple in toy_dataset:
        s = triple["subject"]
        o = triple["object"]

        s_id = resolved_entities.get(s)
        o_id = resolved_entities.get(o)

        if s_id:
            pred_clusters[s_id].add(s)

        if o_id:
            pred_clusters[o_id].add(o)


    print("\nPredicted Clusters:")
    for cluster_id, entities in pred_clusters.items():
        print(f"{cluster_id}: {entities}")

    # -------------------------
    # Build pairwise links
    # -------------------------
    gold_links = set()
    for entities in gold_clusters.values():
        gold_links.update(pairwise_links(entities))

    pred_links = set()
    for entities in pred_clusters.values():
        pred_links.update(pairwise_links(entities))

    # -------------------------
    # Compute metrics
    # -------------------------
    true_positive = len(gold_links & pred_links)
    false_positive = len(pred_links - gold_links)
    false_negative = len(gold_links - pred_links)

    precision = true_positive / (true_positive + false_positive + 1e-10)
    recall = true_positive / (true_positive + false_negative + 1e-10)
    f1 = 2 * precision * recall / (precision + recall + 1e-10)

    print("\nEvaluation Results:")
    print(f"Pairwise Precision: {precision:.3f}")
    print(f"Pairwise Recall   : {recall:.3f}")
    print(f"Pairwise F1       : {f1:.3f}")

    # -------------------------
    # Problematic merges (over-merging)
    # -------------------------

    print(evaluate_cluster_alignment(gold_clusters, pred_clusters))

In [6]:
resolved_entities = resolver.get_text_to_entity_mapping()
    
print(resolved_entities)

{'Apple': 'E5', 'iPhone 15': 'E2', 'September': 'E4', 'Fruit': 'E6', 'Apple Inc.': 'E7', 'MacBook': 'E8'}


In [7]:
evaluate_er_pipeline(resolved_entities, toy_data)

Gold Clusters:
APPLE_TECH: {'Apple Inc.', 'Apple'}
IPHONE_15: {'iPhone 15'}
APPLE_FRUIT: {'Apple'}
MONTH_SEPTEMBER: {'September'}
FRUIT: {'Fruit'}
MACBOOK: {'MacBook'}

Predicted Clusters:
E5: {'Apple'}
E2: {'iPhone 15'}
E4: {'September'}
E6: {'Fruit'}
E7: {'Apple Inc.'}
E8: {'MacBook'}

Evaluation Results:
Pairwise Precision: 0.833
Pairwise Recall   : 0.833
Pairwise F1       : 0.833
Predicted Cluster: E5
Entities: {'Apple'}
Mixed cluster! Matches multiple gold clusters:
  - APPLE_FRUIT: {'Apple'}
  - APPLE_TECH: {'Apple'}
Predicted Cluster: E2
Entities: {'iPhone 15'}
Pure match with gold cluster: IPHONE_15
Predicted Cluster: E4
Entities: {'September'}
Pure match with gold cluster: MONTH_SEPTEMBER
Predicted Cluster: E6
Entities: {'Fruit'}
Pure match with gold cluster: FRUIT
Predicted Cluster: E7
Entities: {'Apple Inc.'}
Pure match with gold cluster: APPLE_TECH
Predicted Cluster: E8
Entities: {'MacBook'}
Pure match with gold cluster: MACBOOK
None
